# OASIS-2 - MRI-derived and longitudinal predictors of dementia
Cleaned analysis notebook accompanying the thesis. Run top to bottom.

**Pipeline:** load data -> extract MRI proxy features -> build annualised change features -> methodology checks and figures -> descriptive and univariate statistics -> models (Targets A and B) -> stability analyses -> SHAP interpretation.


## 1. Setup, data loading and feature definitions


In [ ]:
# Imports
import os, tarfile, tempfile, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from scipy import ndimage

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_validate, cross_val_predict)
from sklearn.metrics import roc_auc_score, confusion_matrix
from skimage.feature import graycomatrix, graycoprops


In [ ]:
# Paths and configuration
# Data lives in the data/ subfolder next to this notebook.
ARCHIVES = [
    'data/OAS2_RAW_PART1.tar.gz',
    'data/OAS2_RAW_PART2.tar.gz',
]
EXCEL_PATH     = 'data/oasis_longitudinal.xlsx'

# MRI features are cached here after the first extraction so the slow
# image-processing step (~30 min) only runs once. Delete this file to
# force a fresh extraction.
MRI_CACHE_PATH = 'data/mri_features_extracted.csv'


In [ ]:
# STEP 1 - Load and prepare the demographic / clinical spreadsheet
print("STEP 1: Loading and preparing demographics")

df = pd.read_excel(EXCEL_PATH)
df = df.sort_values(['Subject ID', 'Visit']).reset_index(drop=True)

# Encode sex numerically: F = 0, M = 1
df['sex'] = (df['M/F'] == 'M').astype(int)

print(f"  {df['Subject ID'].nunique()} subjects ({len(df)} rows)")
print(f"  Group distribution:\n{df['Group'].value_counts().to_string()}")
print(f"  Visits: min={df.groupby('Subject ID').size().min()}  "
      f"max={df.groupby('Subject ID').size().max()}  "
      f"mean={df.groupby('Subject ID').size().mean():.1f}")


In [ ]:
# STEP 2 - Feature Sets 


CLINICAL_CDR     = ['Age', 'sex', 'EDUC', 'SES', 'MMSE', 'CDR']

# Clinical WITHOUT CDR 
CLINICAL_NO_CDR  = ['Age', 'sex', 'EDUC', 'SES', 'MMSE']

# Clinical WITHOUT CDR AND WITHOUT MMSE
# (tests whether model works without any cognitive score)
CLINICAL_NO_COG  = ['Age', 'sex', 'EDUC', 'SES']

FREESURFER  = ['eTIV', 'nWBV', 'ASF']

# MRI-derived features from raw images (honest proxy labels)
MRI_STATIC = [
    'csf_brain_ratio',    # CSF/brain proxy for ventricular enlargement
    'gw_proxy',           # threshold-based grey/white intensity proxy
    'asymmetry',          # left-right hemisphere intensity asymmetry
    'glcm_contrast',      # texture contrast (exploratory)
    'glcm_homogeneity',   # texture homogeneity (exploratory)
    'glcm_entropy',       # texture entropy (exploratory)
]
 
# Annualized delta features — delta_CDR excluded (too close to label)
MRI_DELTA = [
    'delta_MMSE_annual',      # cognitive decline rate (from CSV)
    'delta_nWBV_annual_csv',  # brain atrophy rate (FreeSurfer CSV)
    'delta_csf_annual',       # CSF ratio change rate (raw image)
    'delta_gw_annual',        # grey/white proxy change rate (raw image)
    'delta_asym_annual',      # asymmetry change rate (raw image)
]
# Target A feature sets 
FEATURE_SETS_A = {
    'S1_Clin':           CLINICAL_NO_CDR,
    'S2_Clin_FS':        CLINICAL_NO_CDR + FREESURFER,
    'S3_Clin_MRI':       CLINICAL_NO_CDR + MRI_STATIC,
    'S4_Clin_FS_MRI':    CLINICAL_NO_CDR + FREESURFER + MRI_STATIC,
    
    'S5_NoCog_FS':       CLINICAL_NO_COG + FREESURFER,
    'S6_NoCog_MRI':      CLINICAL_NO_COG + MRI_STATIC,
    'S7_NoCog_FS_MRI':   CLINICAL_NO_COG + FREESURFER + MRI_STATIC,
    
    'S2_CDR_FS':         CLINICAL_CDR    + FREESURFER,
    'S4_CDR_FS_MRI':     CLINICAL_CDR    + FREESURFER + MRI_STATIC,
}

#Target B feature sets similarly
FEATURE_SETS_B1 = {
    'B1_Clin':           CLINICAL_NO_CDR,
    'B1_Clin_FS':        CLINICAL_NO_CDR + FREESURFER,
    'B1_Clin_MRI':       CLINICAL_NO_CDR + MRI_STATIC,
    'B1_Clin_FS_MRI':    CLINICAL_NO_CDR + FREESURFER + MRI_STATIC,
}

FEATURE_SETS_B2 = {
    'B2_Clin_delta':     CLINICAL_NO_CDR + MRI_DELTA,
    'B2_Clin_FS_delta':  CLINICAL_NO_CDR + FREESURFER + MRI_DELTA,
    'B2_Clin_MRI_delta': CLINICAL_NO_CDR + MRI_STATIC + MRI_DELTA,
    'B2_All_delta':      CLINICAL_NO_CDR + FREESURFER + MRI_STATIC + MRI_DELTA,
}

COLORS = {
    'S1_Clin':          '#888888',
    'S2_Clin_FS':       '#4C72B0',
    'S3_Clin_MRI':      '#DD8452',
    'S4_Clin_FS_MRI':   '#55A868',
    'S2_CDR_FS':        '#9ecae1',
    'S4_CDR_FS_MRI':    '#a1d99b',
    'B1_Clin':          '#888888',
    'B1_Clin_FS':       '#4C72B0',
    'B1_Clin_MRI':      '#DD8452',
    'B1_Clin_FS_MRI':   '#55A868',
    'B2_Clin_delta':    '#888888',
    'B2_Clin_FS_delta': '#4C72B0',
    'B2_Clin_MRI_delta':'#DD8452',
    'B2_All_delta':     '#55A868',
}
 
COLORS.update({
    'S5_NoCog_FS':        '#9467bd',
    'S6_NoCog_MRI':       '#e377c2',
    'S7_NoCog_FS_MRI':    '#bcbd22',
})


## 2. MRI proxy feature extraction and annualised change features


In [ ]:
# STEP 3 — MRI FEATURE EXTRACTION
print("=" * 70)
print("STEP 3: MRI feature extraction")
print("=" * 70)


def extract_mri_features(data):
    """
    Extract features from one averaged 3D MRI scan.
    Uses p99 threshold (not max) to handle atypical low-intensity scans.
    Uses np.squeeze to handle (128,256,256,1) shaped scans.
    """
    data = np.squeeze(data)
    p99  = np.percentile(data, 99)

    skull_mask = ndimage.binary_fill_holes(data > p99 * 0.05)
    csf_mask   = (skull_mask &
                  (data < p99 * 0.15) &
                  (data > p99 * 0.02))
    brain_mask = ndimage.binary_opening(
                    skull_mask & (data > p99 * 0.25),
                    iterations=2)
    white_mask = skull_mask & (data > p99 * 0.45)
    grey_mask  = (skull_mask &
                  (data > p99 * 0.25) &
                  (data <= p99 * 0.45))

    if brain_mask.sum() < 1000:
        csf_brain_ratio = np.nan
        gw_proxy        = np.nan
    else:
        csf_brain_ratio = float(np.clip(
            csf_mask.sum() / (brain_mask.sum() + 1e-8), 0, 5))
        gw_proxy        = float(np.clip(
            grey_mask.sum() / (white_mask.sum() + 1e-8), 0, 50))

    mid           = data.shape[0] // 2
    right_hemi    = data[:mid]
    left_hemi     = data[mid:]
    right_flipped = right_hemi[::-1]
    s             = min(left_hemi.shape[0], right_flipped.shape[0])
    asymmetry     = float(
        np.abs(left_hemi[:s] - right_flipped[:s]).mean()
        / (data.mean() + 1e-8)
    )

    try:
        mid_axial = data[:, :, data.shape[2]//2] if data.ndim == 3 else data
        img_uint8 = ((mid_axial - mid_axial.min()) /
                     (mid_axial.max() - mid_axial.min() + 1e-8) * 255
                     ).astype(np.uint8)
        glcm = graycomatrix(img_uint8, distances=[1],
                             angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                             levels=256, symmetric=True, normed=True)
        glcm_contrast    = float(graycoprops(glcm, 'contrast').mean())
        glcm_homogeneity = float(graycoprops(glcm, 'homogeneity').mean())
        glcm_flat        = glcm.flatten()
        glcm_flat        = glcm_flat[glcm_flat > 0]
        glcm_entropy     = float(-np.sum(glcm_flat * np.log2(glcm_flat + 1e-10)))
    except Exception:
        glcm_contrast = glcm_homogeneity = glcm_entropy = np.nan

    return {
        'csf_brain_ratio':  csf_brain_ratio,
        'gw_proxy':         gw_proxy,
        'asymmetry':        float(asymmetry),
        'glcm_contrast':    glcm_contrast,
        'glcm_homogeneity': glcm_homogeneity,
        'glcm_entropy':     glcm_entropy,
    }


def extract_all_from_archives(df_in, archives, cache_path):
    """
    Open each archive ONCE. For each scan:
      1. Load mpr-1, mpr-2, mpr-3 repeats
      2. Reorient each to RAS+ canonical orientation
      3. Verify orientation is ('R','A','S')
      4. Average the repeats (improves SNR)
      5. Extract features from averaged volume
    Saves to CSV cache so this only runs once.
    """
    # Load from cache if available
    if os.path.exists(cache_path):
        print(f"  Loading cached features from:\n  {cache_path}")
        mri_df    = pd.read_csv(cache_path)
        matched   = df_in['MRI ID'].isin(mri_df['MRI ID']).sum()
        unmatched = (~df_in['MRI ID'].isin(mri_df['MRI ID'])).sum()
        print(f"  Loaded {len(mri_df)} cached scan features")
        print(f"  Matched to spreadsheet: {matched}/{len(df_in)}")
        if unmatched > 0:
            print(f"  Warning: {unmatched} scans not in cache")
        return mri_df

    print("  No cache — extracting (averages mpr-1/2/3, ~30 min)...")
    mri_ids              = df_in['MRI ID'].tolist()
    all_records          = []
    done                 = 0
    t0                   = time.time()
    total                = len(mri_ids)
    orientation_warnings = []

    for archive_path in archives:
        print(f"\n  Opening {os.path.basename(archive_path)}...")
        with tarfile.open(archive_path, "r:gz") as tar:

            # Build index once: "OAS2_0100_MR1_mpr-1" → (hdr_m, img_m)
            hdr_index = {}
            for m in tar.getmembers():
                if not m.name.endswith('.hdr'):
                    continue
                mpr = next((f'mpr-{i}' for i in [1,2,3]
                            if f'mpr-{i}' in m.name), None)
                if not mpr:
                    continue
                parts = m.name.split('/')
                if len(parts) < 2:
                    continue
                mri_id = parts[1]
                try:
                    img_m = tar.getmember(m.name.replace('.hdr','.img'))
                    hdr_index[f"{mri_id}_{mpr}"] = (m, img_m)
                except KeyError:
                    pass

            n_primary = len([k for k in hdr_index if 'mpr-1' in k])
            print(f"  Index: {n_primary} primary scans, "
                  f"{len(hdr_index)} total acquisitions\n")

            for mri_id in mri_ids:
                if f"{mri_id}_mpr-1" not in hdr_index:
                    continue
                done += 1

                try:
                    volumes      = []
                    orientations = []

                    for mpr in ['mpr-1', 'mpr-2', 'mpr-3']:
                        key = f"{mri_id}_{mpr}"
                        if key not in hdr_index:
                            continue
                        hdr_m, img_m = hdr_index[key]
                        try:
                            with tempfile.TemporaryDirectory() as tmp:
                                tar.extract(hdr_m, path=tmp, filter='data')
                                tar.extract(img_m, path=tmp, filter='data')
                                hdr_path = os.path.join(tmp, hdr_m.name)

                                # mmap=False fixes WinError 32 on Windows
                                img    = nib.load(hdr_path, mmap=False)
                                img    = nib.as_closest_canonical(img)
                                orient = nib.aff2axcodes(img.affine)
                                orientations.append(orient)

                                if orient != ('R', 'A', 'S'):
                                    orientation_warnings.append(
                                        f"{mri_id} {mpr}: {orient}")

                                vol = np.array(
                                    img.get_fdata(caching='unchanged'))
                                img.uncache()
                                del img
                                volumes.append(np.squeeze(vol))
                        except Exception:
                            pass

                    if not volumes:
                        continue

                    # Average repeats
                    if len(volumes) == 1:
                        data = volumes[0]
                    else:
                        shapes = [v.shape for v in volumes]
                        if len(set(shapes)) == 1:
                            data = np.mean(np.stack(volumes, axis=0), axis=0)
                        else:
                            data = volumes[0]

                    # Log first scan
                    if done == 1 and orientations:
                        print(f"  First scan: {mri_id}")
                        print(f"  Orientation: {orientations[0]}")
                        print(f"  Repeats averaged: {len(volumes)}/3")
                        print(f"  Shape: {data.shape}\n")

                    feats           = extract_mri_features(data)
                    feats['MRI ID'] = mri_id
                    all_records.append(feats)

                except Exception as e:
                    print(f"\n  Warning [{mri_id}]: {e}")

                elapsed   = time.time() - t0
                rate      = done / elapsed if elapsed > 0 else 1
                remaining = (total - done) / rate / 60
                print(f"  [{done}/{total}] {mri_id}  "
                      f"~{remaining:.0f} min left     ", end='\r')

    # Orientation summary
    print(f"\n\n  Orientation verification:")
    if orientation_warnings:
        print(f"  WARNING: {len(orientation_warnings)} non-RAS+ scans:")
        for w in orientation_warnings:
            print(f"    {w}")
    else:
        print(f"  All {done} scans confirmed RAS+ after reorientation ✓")
        print(f"  Left/right asymmetry split along axis 0 is valid")

    # Save cache
    mri_df = pd.DataFrame(all_records)
    mri_df.to_csv(cache_path, index=False)
    elapsed_min = (time.time() - t0) / 60
    print(f"\n  Done in {elapsed_min:.1f} min — "
          f"{len(mri_df)}/{total} scans extracted")
    print(f"  Saved to: {cache_path}")
    return mri_df


# ── Call the function and create df_full ──────────────────
mri_df  = extract_all_from_archives(df, ARCHIVES, MRI_CACHE_PATH)
df_full = df.merge(mri_df, on='MRI ID', how='inner')
print(f"\n  Combined dataset: {len(df_full)} rows, "
      f"{df_full['Subject ID'].nunique()} subjects")

# Sanity check
print("\n  Feature sanity check:")
for col in ['csf_brain_ratio', 'gw_proxy', 'asymmetry',
            'glcm_entropy', 'glcm_homogeneity']:
    if col in df_full.columns:
        print(f"    {col:<22}  "
              f"min={df_full[col].min():.4f}  "
              f"max={df_full[col].max():.4f}  "
              f"NaN={df_full[col].isna().sum()}")


In [ ]:
# STEP 4 — DELTA FEATURES (visit 1 → visit 2)

print("STEP 4: Computing annualized delta features (visit 1 → visit 2)")
 
def compute_deltas(df_in):
    """
    Build one row of annualised change features per subject (visit v1 -> v2).

    Conversion timing for the Converted group (CDR-based, as in Section 4.2):
      v2 (conversion visit) = first visit at which CDR > 0.
      v1 (baseline visit)   = last visit before v2 at which CDR was 0.
    Stable Nondemented subjects use visit 1 -> visit 2.
    delta_CDR is computed for reporting only and is never used as a predictor.
    """
    EXCLUDE = ['OAS2_0131']
    all_cols = (CLINICAL_CDR + FREESURFER + MRI_STATIC + ['Group','MRI ID'])
    records  = []

    for subj_id, group in df_in.groupby('Subject ID'):
        if subj_id in EXCLUDE:
            continue

        group    = group.sort_values('Visit').reset_index(drop=True)
        subj_grp = group.iloc[0]['Group']

        if subj_grp == 'Converted':
            # v2 = first visit with CDR > 0; v1 = last preceding visit with CDR 0
            last_ndem_visit  = None
            conversion_visit = None
            for _, row in group.iterrows():
                if row['CDR'] == 0:
                    last_ndem_visit = row      # still nondemented
                else:                          # CDR > 0
                    conversion_visit = row     # first visit with CDR > 0
                    break
            if last_ndem_visit is None or conversion_visit is None:
                continue
            v1 = last_ndem_visit
            v2 = conversion_visit

        elif subj_grp == 'Nondemented':
            if len(group) < 2:
                continue
            v1 = group.iloc[0]
            v2 = group.iloc[1]
        else:
            continue

        time_gap = v2['Age'] - v1['Age']
        if time_gap < 0.3:
            continue

        rec = {
            'Subject ID': subj_id,
            'time_gap':   time_gap,
            'v1_visit':   v1['Visit'],
            'v2_visit':   v2['Visit'],
        }

        for col in all_cols:
            if col in v1.index:
                rec[col] = v1[col]

        rec['delta_nWBV_annual_csv'] = (
            (v2['nWBV'] - v1['nWBV']) / time_gap)
        rec['delta_MMSE_annual']     = (
            (v2['MMSE'] - v1['MMSE']) / time_gap)

        for raw_col, delta_key in [
            ('csf_brain_ratio', 'delta_csf_annual'),
            ('gw_proxy',        'delta_gw_annual'),
            ('asymmetry',       'delta_asym_annual'),
        ]:
            if raw_col in v1.index and raw_col in v2.index:
                rec[delta_key] = (
                    (v2[raw_col] - v1[raw_col]) / time_gap)
            else:
                rec[delta_key] = np.nan

        rec['delta_CDR_REPORTONLY'] = (
            (v2['CDR'] - v1['CDR']) / time_gap)

        records.append(rec)

    return pd.DataFrame(records)
 
df_delta = compute_deltas(df_full)
print(f"  Subjects with delta features: {len(df_delta)}")
print(f"  Mean time gap v1→v2: {df_delta['time_gap'].mean():.2f} years  "
      f"(min={df_delta['time_gap'].min():.2f}  "
      f"max={df_delta['time_gap'].max():.2f})")


## 3. Methodology checks
Sanity checks that justify the design choices described in the thesis: the OAS2_0131 exclusion, the CDR-based conversion timing, the baseline visit selection, and the RAS+ reorientation.


In [ ]:
print(df[df['Subject ID']=='OAS2_0131'][
    ['Visit','Age','CDR','MMSE','Group']
].to_string())


In [ ]:
EXCLUDE = ['OAS2_0131']
cdr_only = mmse_only = both = 0
rows = []
for subj_id, group in df_full.groupby('Subject ID'):
    if subj_id in EXCLUDE:
        continue
    group = group.sort_values('Visit').reset_index(drop=True)
    if group.iloc[0]['Group'] != 'Converted':
        continue
    baseline_mmse = group.iloc[0]['MMSE']
    for _, row in group.iterrows():
        mmse_drop = baseline_mmse - row['MMSE']
        if row['CDR'] == 0 and mmse_drop < 3:
            continue
        cdr_fired, mmse_fired = row['CDR'] > 0, mmse_drop >= 3
        if cdr_fired and mmse_fired:
            both += 1; tag = 'both'
        elif mmse_fired:
            mmse_only += 1; tag = 'MMSE only'
        else:
            cdr_only += 1; tag = 'CDR only'
        rows.append((subj_id, int(row['Visit']), row['CDR'], round(mmse_drop,1), tag))
        break

print(f"classified converters : {cdr_only + mmse_only + both}")
print(f"  CDR only            : {cdr_only}")
print(f"  MMSE only           : {mmse_only}")
print(f"  both                : {both}")
print(f"  MMSE rule fired     : {mmse_only + both}")
for r in rows: print(r)


In [ ]:
df_delta = compute_deltas(df_full)

print("Verification — which visits used as baseline:")
print(f"{'Subject':<20} {'Group':<14} {'v1_visit':>10} "
      f"{'v2_visit':>10} {'time_gap':>10}")
print("-" * 60)
for _, row in df_delta[
        df_delta['Group']=='Converted'].iterrows():
    print(f"  {row['Subject ID']:<18} "
          f"{row['Group']:<14} "
          f"{row['v1_visit']:>10.0f} "
          f"{row['v2_visit']:>10.0f} "
          f"{row['time_gap']:>10.2f}")

print(f"\nNondemented — first 5 subjects:")
for _, row in df_delta[
        df_delta['Group']=='Nondemented'].head(5).iterrows():
    print(f"  {row['Subject ID']:<18} "
          f"{row['Group']:<14} "
          f"{row['v1_visit']:>10.0f} "
          f"{row['v2_visit']:>10.0f} "
          f"{row['time_gap']:>10.2f}")


In [ ]:
# Sanity check: for every Converted subject, is v1 strictly younger than v2?
conv = df_delta[df_delta['Group'] == 'Converted'].copy()

# Recover v1 and v2 ages by looking each visit up in df_full
def ages_for(row):
    s = df_full[df_full['Subject ID'] == row['Subject ID']]
    a1 = s.loc[s['Visit'] == row['v1_visit'], 'Age'].values
    a2 = s.loc[s['Visit'] == row['v2_visit'], 'Age'].values
    return (a1[0] if len(a1) else np.nan,
            a2[0] if len(a2) else np.nan)

conv[['v1_age', 'v2_age']] = conv.apply(
    lambda r: pd.Series(ages_for(r)), axis=1)
conv['v1_younger'] = conv['v1_age'] < conv['v2_age']

print(conv[['Subject ID', 'v1_visit', 'v2_visit',
            'v1_age', 'v2_age', 'time_gap', 'v1_younger']]
      .to_string(index=False))

n_bad = (~conv['v1_younger']).sum()
print(f"\nConverted subjects where v1 is NOT younger than v2: {n_bad}")
assert n_bad == 0, "Some converted subjects have v1_age >= v2_age!"
print("All converted subjects: v1 age < v2 age ✓")


In [ ]:
import tarfile, tempfile, nibabel as nib, numpy as np

print("Checking orientation across multiple scans...")
print("(using as_closest_canonical — should all be RAS+)\n")

# Check first scan from each archive + a few random ones
check_ids = ['OAS2_0001_MR1',   # first in archive 1
             'OAS2_0100_MR1',   # middle
             'OAS2_0186_MR3',   # last
             'OAS2_0018_MR1',   # Converted patient
             'OAS2_0014_MR1',]  # was a NaN scan

results = {}
for mri_id in check_ids:
    for archive in ARCHIVES:
        try:
            with tarfile.open(archive, "r:gz") as tar:
                hdrs = [m for m in tar.getmembers()
                        if mri_id in m.name
                        and m.name.endswith('.hdr')
                        and 'mpr-1' in m.name]
                if not hdrs:
                    continue
                hdr_m = hdrs[0]
                img_m = tar.getmember(hdr_m.name.replace('.hdr','.img'))
                with tempfile.TemporaryDirectory() as tmp:
                    tar.extract(hdr_m, path=tmp, filter='data')
                    tar.extract(img_m, path=tmp, filter='data')

                    # Check BEFORE canonical reorientation
                    img_raw = nib.load(
                        os.path.join(tmp, hdr_m.name), mmap=False)
                    orient_before = nib.aff2axcodes(img_raw.affine)

                    # Check AFTER canonical reorientation
                    img_can = nib.as_closest_canonical(img_raw)
                    orient_after  = nib.aff2axcodes(img_can.affine)

                    shape_before = img_raw.shape
                    shape_after  = np.squeeze(
                        img_can.get_fdata(caching='unchanged')).shape

                    img_raw.uncache(); del img_raw
                    img_can.uncache(); del img_can

                results[mri_id] = {
                    'before': orient_before,
                    'after':  orient_after,
                    'shape_before': shape_before,
                    'shape_after':  shape_after,
                }
                break
        except Exception as e:
            results[mri_id] = {'error': str(e)}

print(f"{'MRI ID':<20} {'Before canonical':>18} {'After canonical':>18} "
      f"{'Shape after':>14} {'OK?':>5}")
print("-" * 80)
for mri_id, r in results.items():
    if 'error' in r:
        print(f"  {mri_id:<18} ERROR: {r['error']}")
        continue
    ok = '✓' if r['after'] == ('R','A','S') else '✗ PROBLEM'
    print(f"  {mri_id:<18} {str(r['before']):>18} "
          f"{str(r['after']):>18} "
          f"{str(r['shape_after']):>14} {ok:>5}")

print("\nConclusion:")
all_ok = all(
    r.get('after') == ('R','A','S')
    for r in results.values()
    if 'error' not in r
)
if all_ok:
    print("  All checked scans are in RAS+ orientation after reorientation")
    print("  Left/right asymmetry split along axis 0 is valid")
else:
    print("  WARNING: some scans not in RAS+ — check manually")


## 4. Methodology figures
Figures 1-3: the RAS+ montage, the proxy-feature explainer, and the representative low- vs high-CSF scans.


In [ ]:
# ── Shared helper: load one scan by MRI ID, reoriented to RAS+ ──
import tarfile, tempfile, os
import numpy as np, nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage

def load_ras_volume(mri_id, archives=ARCHIVES, average=False):
    """Return the RAS+ canonical volume for one MRI ID.
    average=True averages mpr-1/2/3 (matches extraction); False = mpr-1 only (faster)."""
    mprs = ['mpr-1', 'mpr-2', 'mpr-3'] if average else ['mpr-1']
    for archive in archives:
        with tarfile.open(archive, 'r:gz') as tar:
            if not any(mri_id in m.name and m.name.endswith('.hdr') and 'mpr-1' in m.name
                       for m in tar.getmembers()):
                continue
            vols = []
            for mpr in mprs:
                cand = [m for m in tar.getmembers()
                        if mri_id in m.name and m.name.endswith('.hdr') and mpr in m.name]
                if not cand:
                    continue
                hdr_m = cand[0]
                img_m = tar.getmember(hdr_m.name.replace('.hdr', '.img'))
                with tempfile.TemporaryDirectory() as tmp:
                    tar.extract(hdr_m, path=tmp, filter='data')
                    tar.extract(img_m, path=tmp, filter='data')
                    can = nib.as_closest_canonical(
                        nib.load(os.path.join(tmp, hdr_m.name), mmap=False))
                    vols.append(np.squeeze(np.asarray(can.get_fdata(caching='unchanged'))))
            if not vols:
                continue
            return np.mean(vols, axis=0) if len({v.shape for v in vols}) == 1 else vols[0]
    raise FileNotFoundError(mri_id)


In [ ]:
mri_id = 'OAS2_0100_MR1'                       # any clear scan
data   = load_ras_volume(mri_id, average=False)
m0, m1, m2 = [s // 2 for s in data.shape]

fig, ax = plt.subplots(1, 3, figsize=(12, 4.4))
fig.suptitle(f'Canonical RAS+ orientation — {mri_id}\n'
             'Axis 0 = Right→Left   Axis 1 = Posterior→Anterior   Axis 2 = Inferior→Superior',
             fontsize=10, fontweight='bold')

for a, sl, ttl in zip(ax,
        [data[m0, :, :], data[:, m1, :], data[:, :, m2]],
        ['Sagittal (axis 0 fixed)', 'Coronal (axis 1 fixed)', 'Axial (axis 2 fixed)']):
    a.imshow(sl.T, cmap='gray', origin='lower', aspect='auto')
    a.set_title(ttl, fontsize=9); a.set_xticks([]); a.set_yticks([])

# Overlay the hemispheric split used by the asymmetry feature
ax[2].axvline(data.shape[0] / 2, color='#ff3b3b', ls='--', lw=1.6)
ax[2].text(0.02, 0.97, 'R', transform=ax[2].transAxes, color='w', fontweight='bold', va='top')
ax[2].text(0.95, 0.97, 'L', transform=ax[2].transAxes, color='w', fontweight='bold', va='top')
ax[2].set_title('Axial + left/right asymmetry split', fontsize=9)
plt.tight_layout()
plt.savefig(r'fig_ras_montage.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── Cell M2 (fixed): proxy-feature explainer ────────────────────
mri_id = 'OAS2_0100_MR1'
Z_FRAC = 0.55                       # 0.5 = mid-slice; raise slightly toward the ventricles
data = load_ras_volume(mri_id, average=True)
p99  = np.percentile(data, 99)
z    = int(data.shape[2] * Z_FRAC)

# 3D masks — identical thresholds to extract_mri_features
skull = ndimage.binary_fill_holes(data > p99*0.05)
csf   = skull & (data < p99*0.15) & (data > p99*0.02)
brain = ndimage.binary_opening(skull & (data > p99*0.25), iterations=2)
white = skull & (data > p99*0.45)
grey  = skull & (data > p99*0.25) & (data <= p99*0.45)

# Pull TRUE feature values so the figure matches the thesis tables
row = df_full.loc[df_full['MRI ID'] == mri_id].iloc[0]
csf_v, gw_v, asym_v = row['csf_brain_ratio'], row['gw_proxy'], row['asymmetry']

base = data[:, :, z]
base = (base - base.min()) / (base.max() - base.min() + 1e-8)
def rgb(): return np.dstack([base]*3)
csf_o = rgb(); csf_o[csf[:, :, z]] = [0.1, 0.5, 1.0]
gw_o  = rgb(); gw_o[grey[:, :, z]] = [0.95, 0.5, 0.05]; gw_o[white[:, :, z]] = [1, 1, 0.2]

# Asymmetry: mirror the half-difference back to full width, overlay on anatomy
m0    = base.shape[0] // 2
right = base[:m0]; left = base[m0:2*m0]
s     = min(right.shape[0], left.shape[0])
dhalf = np.abs(left[:s] - right[::-1][:s])
amap  = np.zeros_like(base)
amap[m0:m0+s] = dhalf            # left half
amap[m0-s:m0] = dhalf[::-1]      # mirror onto right half

fig, ax = plt.subplots(2, 3, figsize=(13.5, 8.6))
fig.suptitle(f'MRI-derived proxy features on {mri_id} (axial slice)',
             fontsize=12, fontweight='bold')

ax[0,0].imshow(base.T, cmap='gray', origin='lower')
ax[0,0].set_title('Reoriented MRI (input)', fontsize=9)
ax[0,1].imshow(csf_o.transpose(1,0,2), origin='lower')
ax[0,1].set_title(f'csf_brain_ratio = {csf_v:.3f}\n(blue = CSF-like voxels)', fontsize=9)
ax[0,2].imshow(gw_o.transpose(1,0,2), origin='lower')
ax[0,2].set_title(f'gw_proxy = {gw_v:.3f}\n(orange = grey, yellow = white)', fontsize=9)

ax[1,0].imshow(base.T, cmap='gray', origin='lower')
ax[1,0].imshow(amap.T, cmap='hot', origin='lower', alpha=0.55)
ax[1,0].axvline(m0, color='cyan', ls='--', lw=1.2)
ax[1,0].set_title(f'asymmetry = {asym_v:.3f}\n(bright = L/R mismatch)', fontsize=9)

ax[1,1].imshow(base.T, cmap='gray', origin='lower')
ax[1,1].set_title('GLCM texture input\n(8-bit, dist 1, 4 angles)', fontsize=9)

from skimage.feature import graycomatrix
u8 = (base*255).astype(np.uint8)
g = graycomatrix(u8, [1], [0], levels=256, symmetric=True, normed=True)[:, :, 0, 0]
g[0, 0] = 0                      # suppress the dominant background-background pair

LO, HI = 0, 40                  # zoom window — raise LO to ~10 if the corner still saturates
ax[1,2].imshow(np.log1p(g[LO:HI, LO:HI]), cmap='magma', origin='lower',
               extent=[LO, HI, LO, HI])
ax[1,2].set_title('GLCM (log scale, zoomed)', fontsize=9)
ax[1,2].set_xlabel('grey level j'); ax[1,2].set_ylabel('grey level i')
for a in ax.ravel():
    if a is not ax[1,2]:
        a.axis('off')
plt.tight_layout()
plt.savefig(r'fig_proxy_explainer.pdf', bbox_inches='tight')
plt.show()


In [ ]:
nd = df_full[df_full['Group']=='Nondemented'].sort_values('csf_brain_ratio')
dm = df_full[df_full['Group']=='Demented'].sort_values('csf_brain_ratio')
nd_id = nd.iloc[len(nd)//4]['MRI ID']      # ~25th pct CSF
dm_id = dm.iloc[3*len(dm)//4]['MRI ID']    # ~75th pct CSF

fig, ax = plt.subplots(2, 2, figsize=(9, 9))
fig.suptitle('Representative low-CSF (Nondemented) vs high-CSF (Demented) scans',
             fontsize=12, fontweight='bold')
for col, (mid_id, label) in enumerate([(nd_id,'Nondemented'), (dm_id,'Demented')]):
    d = load_ras_volume(mid_id, average=True)
    p99 = np.percentile(d, 99); z = d.shape[2]//2
    skull = ndimage.binary_fill_holes(d > p99*0.05)
    csf   = skull & (d < p99*0.15) & (d > p99*0.02)
    b = d[:,:,z]; b = (b-b.min())/(b.max()-b.min()+1e-8); o = np.dstack([b]*3)
    o[csf[:,:,z]] = [0.1,0.5,1.0]
    ax[0,col].imshow(b.T, cmap='gray', origin='lower'); ax[0,col].axis('off')
    ax[0,col].set_title(f'{label}\n{mid_id}', fontsize=10, fontweight='bold')
    ax[1,col].imshow(o.transpose(1,0,2), origin='lower'); ax[1,col].axis('off')
    ax[1,col].set_title('CSF-like voxels (blue)', fontsize=9)
plt.tight_layout()
plt.savefig(r'fig_dem_vs_nondem.pdf', bbox_inches='tight')
plt.show()


## 5. Descriptive and univariate statistics


In [ ]:
desc_vars = [
    "Age", "sex", "EDUC", "SES", "MMSE", "CDR",
    "eTIV", "nWBV", "ASF",
    "csf_brain_ratio", "gw_proxy", "asymmetry",
    "glcm_contrast", "glcm_homogeneity", "glcm_entropy"
]

var_labels = {
    "Age": "Age",
    "sex": "Sex (male = 1)",
    "EDUC": "Education",
    "SES": "SES",
    "MMSE": "MMSE",
    "CDR": "CDR",
    "eTIV": "eTIV",
    "nWBV": "nWBV",
    "ASF": "ASF",
    "csf_brain_ratio": "CSF-to-brain ratio",
    "gw_proxy": "Grey-white proxy",
    "asymmetry": "Asymmetry",
    "glcm_contrast": "GLCM contrast",
    "glcm_homogeneity": "GLCM homogeneity",
    "glcm_entropy": "GLCM entropy"
}

summary = pd.DataFrame({
    "Variable": [var_labels[v] for v in desc_vars],
    "Mean": [df_full[v].mean() for v in desc_vars],
    "Std. dev.": [df_full[v].std() for v in desc_vars],
    "Min": [df_full[v].min() for v in desc_vars],
    "Max": [df_full[v].max() for v in desc_vars],
    "Missing": [df_full[v].isna().sum() for v in desc_vars]
})

summary[["Mean", "Std. dev.", "Min", "Max"]] = summary[
    ["Mean", "Std. dev.", "Min", "Max"]
].round(3)

print(summary.to_latex(index=False, escape=False))


In [ ]:
# STEP 7 - Univariate group comparisons (Mann-Whitney U + Cohen's d)
# Target A -> Table 5 (Demented vs Nondemented, with Cohen's d)
# Target B -> Table 6 (Converted vs stable Nondemented; d omitted, small group)
from scipy.stats import mannwhitneyu

def cohens_d(a, b):
    n1, n2     = len(a), len(b)
    pooled     = np.sqrt(((n1-1)*a.var() + (n2-1)*b.var()) / (n1+n2-2 + 1e-8))
    return (a.mean() - b.mean()) / (pooled + 1e-8)

def pairwise_table(df_in, group_col, g1, g2, test_cols, label, show_d=True):
    d1 = df_in[df_in[group_col] == g1]
    d2 = df_in[df_in[group_col] == g2]
    print(f"\n  {label}")
    print(f"  {g1} (n={len(d1)})  vs  {g2} (n={len(d2)})")
    header = f"  {'Feature':<25} {g1[:10]:>10} {g2[:10]:>12} {'p-value':>10}"
    if show_d:
        header += f" {'Cohen d':>9} {'Effect':>9}"
    print(header)
    print("  " + "-" * (len(header) - 2))
    for col in test_cols:
        if col not in df_in.columns:
            continue
        a, b = d1[col].dropna(), d2[col].dropna()
        if len(a) < 3 or len(b) < 3:
            continue
        _, p = mannwhitneyu(a, b, alternative='two-sided')
        line = f"  {col:<25} {a.mean():>10.3f} {b.mean():>12.3f} {p:>10.4f}"
        if show_d:
            d = cohens_d(a, b)
            effect = ('large'  if abs(d) >= 0.8 else
                      'medium' if abs(d) >= 0.5 else
                      'small'  if abs(d) >= 0.2 else 'negligible')
            line += f" {d:>9.3f} {effect:>9}"
        print(line)

# Target A: Demented vs Nondemented (visit level) -> Table 5
pairwise_table(
    df_full[df_full['Group'].isin(['Demented', 'Nondemented'])],
    'Group', 'Demented', 'Nondemented',
    CLINICAL_CDR + FREESURFER + MRI_STATIC,
    'Target A - Demented vs Nondemented', show_d=True)

# Target B: Converted vs stable Nondemented (baseline + annualised change) -> Table 6
df_stat_b = df_delta[df_delta['Group'].isin(['Converted', 'Nondemented'])].copy()
pairwise_table(
    df_stat_b, 'Group', 'Converted', 'Nondemented',
    CLINICAL_NO_CDR + FREESURFER + MRI_STATIC +
    ['delta_MMSE_annual', 'delta_nWBV_annual_csv',
     'delta_csf_annual', 'delta_CDR_REPORTONLY'],
    'Target B - Converted vs stable Nondemented', show_d=False)


## 6. Predictive models
Target A (Demented vs Nondemented, visit level) and Target B (stable vs Converted, subject level, exploratory).


In [ ]:
# ML HELPERS
# Pipelines (impute -> scale -> classifier), subject-level CV,
# composite model-selection scores, and the per-target runners.

def prepare_X(dataframe, cols):
    """
    Select columns only.
    NaN imputation happens inside the pipeline via SimpleImputer
    to prevent data leakage — median computed from training fold only.
    No fillna here.
    """
    available = [c for c in cols if c in dataframe.columns]
    missing   = [c for c in cols if c not in dataframe.columns]
    if missing:
        print(f"    Note — columns not available: {missing}")
    return dataframe[available].copy()


def subject_level_cv(df_in, y_series, n_splits=5):
    """
    Build CV folds split by Subject ID.
    No patient visits appear in both train and test.
    df_in must have reset_index(drop=True) already applied.
    """
    subjects    = df_in['Subject ID'].values
    unique_s    = np.unique(subjects)
    y_vals      = y_series.values
    subj_labels = np.array([
        pd.Series(y_vals[subjects == s]).mode()[0]
        for s in unique_s
    ])
    skf   = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    folds = []
    for tr, te in skf.split(unique_s, subj_labels):
        folds.append((
            np.where(np.isin(subjects, unique_s[tr]))[0],
            np.where(np.isin(subjects, unique_s[te]))[0],
        ))
    return folds


def make_models():
    """
    Four classifiers — all use SimpleImputer + StandardScaler in pipeline.
    Elastic Net LR replaces plain LR — handles correlated features better.
    HistGradientBoosting supports class_weight unlike GradientBoosting.
    """
    return {
        # Elastic Net: L1+L2 regularization, handles correlated features
        'Elastic Net LR':     Pipeline([
                                ('imputer', SimpleImputer(strategy='median')),
                                ('sc',      StandardScaler()),
                                ('clf',     LogisticRegression(
                                    penalty='elasticnet',
                                    solver='saga',
                                    l1_ratio=0.5,
                                    class_weight='balanced',
                                    max_iter=2000,
                                    random_state=42))]),
        'SVM':                Pipeline([
                                ('imputer', SimpleImputer(strategy='median')),
                                ('sc',      StandardScaler()),
                                ('clf',     SVC(
                                    kernel='rbf',
                                    class_weight='balanced',
                                    probability=True,
                                    random_state=42))]),
        'Random Forest':      Pipeline([
                                ('imputer', SimpleImputer(strategy='median')),
                                ('sc',      StandardScaler()),
                                ('clf',     RandomForestClassifier(
                                    n_estimators=300,
                                    class_weight='balanced',
                                    max_depth=6,
                                    random_state=42))]),
        'Hist Grad Boost':    Pipeline([
                                ('imputer', SimpleImputer(strategy='median')),
                                ('sc',      StandardScaler()),
                                ('clf',     HistGradientBoostingClassifier(
                                    class_weight='balanced',
                                    max_iter=200,
                                    max_depth=3,
                                    learning_rate=0.05,
                                    random_state=42))]),
    }


def run_cv(models, X, y, cv, label):
    """
    Run cross-validation for all models.
    Reports AUC, BalAcc, Sensitivity, Specificity, Precision, F1.
    Specificity computed manually from confusion matrix.
    """
    results = {}
    print(f"\n  {label}")
    print(f"  {'Model':<18} {'AUC':>7} {'BalAcc':>7} "
          f"{'Sens':>7} {'Spec':>7}")
    print("  " + "-" * 48)

    for name, model in models.items():
        r = cross_validate(
            model, X, y, cv=cv,
            scoring=['roc_auc', 'balanced_accuracy',
                     'recall'],          # removed f1, precision
            return_train_score=False
        )
        y_pred = cross_val_predict(model, X, y, cv=cv)
        cm     = confusion_matrix(y, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            specificity = tn / (tn + fp + 1e-8)
        else:
            specificity = np.nan

        r['specificity'] = specificity
        results[name]    = r

        print(f"  {name:<18} "
              f"{r['test_roc_auc'].mean():.4f} "
              f"{r['test_balanced_accuracy'].mean():.4f} "
              f"{r['test_recall'].mean():.4f} "
              f"{specificity:.3f}")
    return results

def composite_score(r, auc_w=0.50, bacc_w=0.50):
    """
    Score_A: equal weight to AUC and balanced accuracy.
    AUC = threshold-independent discrimination.
    BalAcc = (Sens + Spec) / 2, equal weight to both classes.
    No double-counting — BalAcc already summarises both classes.
    Weights sum to 1.
    """
    auc  = r['test_roc_auc'].mean()
    bacc = r['test_balanced_accuracy'].mean()
    return auc_w*auc + bacc_w*bacc


def composite_score_b(r):
    """
    Score_B: for conversion prediction.
    AUC = overall discrimination.
    Sensitivity = catching future converters (high weight).
    Specificity = avoiding false alarms (low weight).
    No double counting — Sens and Spec are independent.
    Weights sum to 1.
    """
    auc  = r['test_roc_auc'].mean()
    sens = r['test_recall'].mean()
    spec = r.get('specificity', 0.5)
    if isinstance(spec, float):
        pass
    elif hasattr(spec, 'mean'):
        spec = float(spec)
    return 0.40*auc + 0.40*sens + 0.20*spec


def best_name(r, target_b=False):
    """
    Select best model by composite score, not AUC alone.
    Falls back to AUC if composite score fails.
    target_b=True uses sensitivity-weighted scoring for Target B.
    """
    score_fn = composite_score_b if target_b else composite_score
    try:
        return max(r, key=lambda n: score_fn(r[n]))
    except Exception:
        return max(r, key=lambda n: r[n]['test_roc_auc'].mean())


def best_auc(r, target_b=False):
    """Return AUC of best model selected by composite score."""
    return r[best_name(r, target_b)]['test_roc_auc'].mean()


def fit_best(results, models_dict, X, y, target_b=False):
    """Fit the composite-score-selected best model on full data."""
    bn = best_name(results, target_b)
    models_dict[bn].fit(X, y)
    return models_dict[bn]


def run_target(feature_sets, df_in, y, cv, label, target_b=False):
    """
    Run all feature sets for one target.
    Returns results dict and fitted models dict.
    target_b=True uses sensitivity-weighted composite score.
    """
    results = {}
    fitted  = {}
    for set_name, feat_cols in feature_sets.items():
        X    = prepare_X(df_in, feat_cols)
        mods = make_models()
        res  = run_cv(mods, X, y, cv, f"{label} — {set_name}")
        results[set_name] = res
        fitted[set_name]  = (
            fit_best(res, mods, X, y, target_b), X
        )
    return results, fitted


def print_feature_set_comparison(results, feature_sets,
                                  ref_key=None, label='',
                                  target_b=False):
    """
    Print comparison table of all feature sets showing:
    composite score, AUC, BalAcc, Sens, Spec.
    Highlights best set by composite score.
    """
    print(f"\n  ── {label}: Feature set comparison ────────────────────")
    print(f"  {'Set':<22} {'Best Model':<18} {'AUC':>7} {'BalAcc':>7} "
          f"{'Sens':>7} {'Spec':>7} {'Score':>8}")
    print("  " + "-" * 78)

    score_fn  = composite_score_b if target_b else composite_score
    ref_auc   = best_auc(results[ref_key], target_b) if ref_key else None
    all_scores = {
        sn: score_fn(results[sn][best_name(results[sn], target_b)])
        for sn in results
    }
    best_set  = max(all_scores, key=all_scores.get)

    for sn in feature_sets:
        if sn not in results:
            continue
        bn    = best_name(results[sn], target_b)
        r     = results[sn][bn]
        auc   = r['test_roc_auc'].mean()
        bacc  = r['test_balanced_accuracy'].mean()
        sens  = r['test_recall'].mean()
        spec  = r.get('specificity', float('nan'))
        if isinstance(spec, np.ndarray):
            spec = float(spec)
        score = all_scores[sn]

        flag = ''
        if sn in ('S2_CDR_FS', 'S4_CDR_FS_MRI'):
            flag = '  ← CEILING'
        elif sn == best_set:
            flag = '  ← BEST (composite)'
        elif sn == 'S4_Clin_FS_MRI':
            flag = '  ← KEY'
        elif sn == ref_key:
            flag = '  ← REFERENCE'

        print(f"  {sn:<22} {bn:<18} {auc:>7.3f} {bacc:>7.3f} "
              f"{sens:>7.3f} {spec:>7.3f} {score:>8.3f}{flag}")


In [ ]:
# STEP 6 — TARGET A: DEMENTED VS NONDEMENTED
print("=" * 70)
print("STEP 6: TARGET A — Demented vs Nondemented (main task)")
print("  Primary: without CDR  |  Ceiling: with CDR")
print("=" * 70)

df_a = df_full[df_full['Group'].isin(['Demented','Nondemented'])].copy()
df_a['label'] = (df_a['Group'] == 'Demented').astype(int)
df_a = df_a.reset_index(drop=True)   # required for subject-level CV

print(f"  Demented: {df_a['label'].sum()}  "
      f"Nondemented: {(df_a['label']==0).sum()}  "
      f"Subjects: {df_a['Subject ID'].nunique()}")

y_a        = df_a['label'].values
cv_folds_a = subject_level_cv(df_a, df_a['label'])

results_a, fitted_a = run_target(
    FEATURE_SETS_A, df_a, y_a, cv_folds_a, 'Target A',
    target_b=False
)

# Show which feature set is best immediately
print_feature_set_comparison(
    results_a, FEATURE_SETS_A,
    ref_key='S2_Clin_FS',
    label='Target A',
    target_b=False
)

# ─────────────────────────────────────────────────────────────
# STEP 7 — TARGET B1: BASELINE CONVERSION PREDICTION
print("\n" + "=" * 70)
print("STEP 7: TARGET B1 — Stable vs Converted, baseline only (exploratory)")
print("  No CDR, no delta features")
print("  NOTE: sensitivity-weighted composite score used for model selection")
print("=" * 70)

df_b = df_delta[df_delta['Group'].isin(['Nondemented','Converted'])].copy()
df_b['label'] = (df_b['Group'] == 'Converted').astype(int)
df_b = df_b.reset_index(drop=True)

print(f"  Nondemented: {(df_b['label']==0).sum()}  "
      f"Converted: {df_b['label'].sum()}  "
      f"(small sample — treat as exploratory)")

y_b  = df_b['label'].values
cv_b = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_b1, fitted_b1 = run_target(
    FEATURE_SETS_B1, df_b, y_b, cv_b, 'Target B1',
    target_b=True   # uses sensitivity-weighted composite score
)

print_feature_set_comparison(
    results_b1, FEATURE_SETS_B1,
    ref_key='B1_Clin_FS',
    label='Target B1',
    target_b=True
)

# ─────────────────────────────────────────────────────────────
# STEP 8 — TARGET B2: EARLY CHANGE CONVERSION PREDICTION
print("\n" + "=" * 70)
print("STEP 8: TARGET B2 — Stable vs Converted, baseline + early change")
print("  delta_CDR excluded to avoid label leakage")
print("=" * 70)

results_b2, fitted_b2 = run_target(
    FEATURE_SETS_B2, df_b, y_b, cv_b, 'Target B2',
    target_b=True
)

print_feature_set_comparison(
    results_b2, FEATURE_SETS_B2,
    ref_key='B2_Clin_FS_delta',
    label='Target B2',
    target_b=True
)


## 7. Stability analyses
Repeated subject-level CV for Target A (Table 8) and Monte Carlo repeated splits for Target B (Table 11).


In [ ]:
# STEP 8B — STABILITY ANALYSIS
print("=" * 70)
print("STEP 8B: Stability analysis")
print("  Target A: Repeated subject-level CV (10×5 = 50 folds)")
print("  Target B: Monte Carlo repeated splits (500 iterations)")
print("=" * 70)

def make_logreg():
    """
    Fixed Elastic Net LR for fair comparison across feature sets.
    Matches make_models() Elastic Net LR exactly.
    """
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('sc',      StandardScaler()),
        ('clf',     LogisticRegression(
            penalty='elasticnet',
            solver='saga',
            l1_ratio=0.5,
            class_weight='balanced',
            max_iter=2000,
            random_state=42))
    ])


# ── Target A: Repeated subject-level CV ───────────────────
print("\n  Target A — Repeated subject-level CV (10×5 = 50 folds)")

def repeated_subject_cv(df_in, y, feature_sets,
                         n_splits=5, n_repeats=10):
    subjects    = df_in['Subject ID'].values
    unique_s    = np.unique(subjects)
    y_vals      = y
    subj_labels = np.array([
        pd.Series(y_vals[subjects == s]).mode()[0]
        for s in unique_s
    ])
    all_aucs = {name: [] for name in feature_sets}
    rskf = RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=n_repeats, random_state=42
    )
    total = n_splits * n_repeats
    for fold_idx, (tr_s, te_s) in enumerate(
            rskf.split(unique_s, subj_labels)):
        train_subj = unique_s[tr_s]
        test_subj  = unique_s[te_s]
        train_idx  = np.where(np.isin(subjects, train_subj))[0]
        test_idx   = np.where(np.isin(subjects, test_subj))[0]
        for set_name, feat_cols in feature_sets.items():
            X = prepare_X(df_in, feat_cols)
            model = make_logreg()
            try:
                model.fit(X.iloc[train_idx], y_vals[train_idx])
                yp = model.predict_proba(X.iloc[test_idx])[:, 1]
                if len(np.unique(y_vals[test_idx])) < 2:
                    continue
                all_aucs[set_name].append(
                    roc_auc_score(y_vals[test_idx], yp)
                )
            except Exception:
                pass
        print(f"  Fold {fold_idx+1}/{total}", end='\r')
    print()
    return all_aucs

sets_a_primary = {k: v for k, v in FEATURE_SETS_A.items()
                  if 'CDR' not in k}

repeated_aucs_a = repeated_subject_cv(
    df_a, y_a, sets_a_primary, n_splits=5, n_repeats=10
)

print(f"\n  {'Set':<22} {'Mean AUC':>10} {'Std':>8} {'95% CI':>18}")
print("  " + "-" * 62)
for sn, aucs in repeated_aucs_a.items():
    if not aucs:
        continue
    arr   = np.array(aucs)
    ci_lo = np.percentile(arr, 2.5)
    ci_hi = np.percentile(arr, 97.5)
    print(f"  {sn:<22} {arr.mean():>10.3f} "
          f"{arr.std():>8.3f}  [{ci_lo:.3f},{ci_hi:.3f}]")

# ΔAUC: S4 vs S2 — main research question
delta_auc_a = None
if ('S4_Clin_FS_MRI' in repeated_aucs_a and
        'S2_Clin_FS' in repeated_aucs_a):
    n  = min(len(repeated_aucs_a['S4_Clin_FS_MRI']),
             len(repeated_aucs_a['S2_Clin_FS']))
    delta_auc_a = (np.array(repeated_aucs_a['S4_Clin_FS_MRI'][:n]) -
                   np.array(repeated_aucs_a['S2_Clin_FS'][:n]))
    print(f"\n  ΔAUC (S4 - S2) — main research question:")
    print(f"  Mean  = {delta_auc_a.mean():+.3f}")
    print(f"  Std   = {delta_auc_a.std():.3f}")
    print(f"  95%CI = [{np.percentile(delta_auc_a,2.5):.3f}, "
          f"{np.percentile(delta_auc_a,97.5):.3f}]")
    print(f"  P(ΔAUC > 0):    {(delta_auc_a > 0).mean():.1%}")
    print(f"  P(ΔAUC > 0.02): {(delta_auc_a > 0.02).mean():.1%}")
    if np.percentile(delta_auc_a, 97.5) < 0.02:
        print("  → MRI features do NOT reliably improve beyond FreeSurfer")
    elif delta_auc_a.mean() > 0.02:
        print("  → MRI features DO reliably improve beyond FreeSurfer")
    else:
        print("  → Improvement is inconsistent across splits")

# MMSE contribution
if ('S2_Clin_FS' in repeated_aucs_a and
        'S5_NoCog_FS' in repeated_aucs_a):
    n  = min(len(repeated_aucs_a['S2_Clin_FS']),
             len(repeated_aucs_a['S5_NoCog_FS']))
    da = (np.array(repeated_aucs_a['S2_Clin_FS'][:n]) -
          np.array(repeated_aucs_a['S5_NoCog_FS'][:n]))
    print(f"\n  MMSE contribution (S2 with MMSE - S5 without MMSE):")
    print(f"  Mean ΔAUC = {da.mean():+.3f}  "
          f"95%CI=[{np.percentile(da,2.5):.3f},"
          f"{np.percentile(da,97.5):.3f}]")
    print(f"  → MMSE adds {da.mean():.3f} AUC points on average")

# ── Target B: Monte Carlo ─────────────────────────────────
print("\n  Target B — Monte Carlo repeated splits (500 iterations)")
print("  70/30 split, min 2 Converted in each test set")

def monte_carlo_splits(df_in, y, feature_sets,
                        n_iter=500, test_size=0.30,
                        min_pos_test=2, random_state=42):
    rng         = np.random.RandomState(random_state)
    subjects    = df_in['Subject ID'].values
    unique_s    = np.unique(subjects)
    y_vals      = y
    subj_labels = np.array([
        pd.Series(y_vals[subjects == s]).mode()[0]
        for s in unique_s
    ])
    pos_s    = unique_s[subj_labels == 1]
    neg_s    = unique_s[subj_labels == 0]
    n_te_pos = max(min_pos_test, int(len(pos_s) * test_size))
    n_te_neg = int(len(neg_s) * test_size)

    res = {name: {'auc':[], 'sensitivity':[], 'specificity':[]}
           for name in feature_sets}
    valid    = 0
    attempts = 0

    while valid < n_iter and attempts < n_iter * 20:
        attempts += 1
        if len(pos_s) < n_te_pos:
            break
        te_pos  = rng.choice(pos_s, n_te_pos, replace=False)
        te_neg  = rng.choice(neg_s, n_te_neg, replace=False)
        te_subj = np.concatenate([te_pos, te_neg])
        tr_subj = unique_s[~np.isin(unique_s, te_subj)]
        tr_labs = subj_labels[np.isin(unique_s, tr_subj)]
        if tr_labs.sum() < 3:
            continue
        tr_idx = np.where(np.isin(subjects, tr_subj))[0]
        te_idx = np.where(np.isin(subjects, te_subj))[0]
        y_tr   = y_vals[tr_idx]
        y_te   = y_vals[te_idx]

        for set_name, feat_cols in feature_sets.items():
            X = prepare_X(df_in, feat_cols)
            model = make_logreg()
            try:
                model.fit(X.iloc[tr_idx], y_tr)
                yp    = model.predict_proba(X.iloc[te_idx])[:, 1]
                ypred = model.predict(X.iloc[te_idx])
                if len(np.unique(y_te)) < 2:
                    continue
                auc = roc_auc_score(y_te, yp)
                cm  = confusion_matrix(y_te, ypred)
                if cm.shape == (2, 2):
                    tn, fp, fn, tp_val = cm.ravel()
                    sens = tp_val / (tp_val + fn  + 1e-8)
                    spec = tn    / (tn     + fp   + 1e-8)
                else:
                    sens = spec = np.nan
                res[set_name]['auc'].append(auc)
                res[set_name]['sensitivity'].append(sens)
                res[set_name]['specificity'].append(spec)
            except Exception:
                pass

        valid += 1
        if valid % 100 == 0:
            print(f"  {valid}/{n_iter} iterations", end='\r')

    print(f"  Completed {valid} valid iterations ({attempts} attempts)")
    return res

# Monte Carlo stability is reported for the annualised-change sets (Table 11)
mc_results = monte_carlo_splits(
    df_b, y_b, FEATURE_SETS_B2, n_iter=500
)

print(f"\n  {'Set':<24} {'AUC':>8} {'±Std':>6} "
      f"{'95% CI':>16} {'Sens':>8} {'Spec':>8}")
print("  " + "-" * 74)
for sn, r in mc_results.items():
    if not r['auc']:
        continue
    aucs = np.array(r['auc'])
    sens = np.array(r['sensitivity'])
    spec = np.array(r['specificity'])
    print(f"  {sn:<24} "
          f"{aucs.mean():>8.3f} "
          f"±{aucs.std():>5.3f} "
          f"[{np.percentile(aucs,2.5):.3f},"
          f"{np.percentile(aucs,97.5):.3f}] "
          f"{sens.mean():>8.3f} "
          f"{spec.mean():>8.3f}")



In [ ]:
# STEP 12 - Final summary
# Headline numbers for the thesis: the S4 vs S2 incremental-value test,
# the MMSE contribution, and the per-set metrics behind Tables 7 and 12.
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

s2_auc = best_auc(results_a['S2_Clin_FS'])
s4_auc = best_auc(results_a['S4_Clin_FS_MRI'])

# ---- Target A: per-set AUC vs the S2 reference (Tables 7 and 12) ----
print(f"\n-- Target A: best model per feature set (AUC vs S2_Clin_FS) --")
print(f"  {'Set':<22} {'Best model':<18} {'AUC':>7} {'vs S2':>7}")
print("  " + "-" * 56)
for sn in FEATURE_SETS_A:
    if sn not in results_a:
        continue
    auc  = best_auc(results_a[sn])
    flag = ('  <- CEILING' if sn in ('S2_CDR_FS', 'S4_CDR_FS_MRI')
            else '  <- KEY' if sn == 'S4_Clin_FS_MRI' else '')
    print(f"  {sn:<22} {best_name(results_a[sn]):<18} "
          f"{auc:.3f}  {auc - s2_auc:+.3f}{flag}")

# ---- Main research question: does S4 beat S2? (repeated CV, Table 8) ----
if 'repeated_aucs_a' in dir() and \
   'S4_Clin_FS_MRI' in repeated_aucs_a and 'S2_Clin_FS' in repeated_aucs_a:
    n  = min(len(repeated_aucs_a['S4_Clin_FS_MRI']),
             len(repeated_aucs_a['S2_Clin_FS']))
    da = (np.array(repeated_aucs_a['S4_Clin_FS_MRI'][:n]) -
          np.array(repeated_aucs_a['S2_Clin_FS'][:n]))
    print(f"\n-- Incremental value of MRI proxy features (S4 - S2) --")
    print(f"  Mean dAUC = {da.mean():+.3f}  Std = {da.std():.3f}  "
          f"95% CI = [{np.percentile(da,2.5):.3f}, {np.percentile(da,97.5):.3f}]")
    print(f"  P(dAUC > 0) = {(da > 0).mean():.1%}   "
          f"P(dAUC > 0.02) = {(da > 0.02).mean():.1%}")

# ---- MMSE contribution: S2 (with MMSE) vs S5 (no cognitive scores) ----
if 'repeated_aucs_a' in dir() and \
   'S2_Clin_FS' in repeated_aucs_a and 'S5_NoCog_FS' in repeated_aucs_a:
    n  = min(len(repeated_aucs_a['S2_Clin_FS']),
             len(repeated_aucs_a['S5_NoCog_FS']))
    dm = (np.array(repeated_aucs_a['S2_Clin_FS'][:n]) -
          np.array(repeated_aucs_a['S5_NoCog_FS'][:n]))
    print(f"\n-- MMSE contribution (S2 with MMSE - S5 without) --")
    print(f"  Mean dAUC = {dm.mean():+.3f}  "
          f"95% CI = [{np.percentile(dm,2.5):.3f}, {np.percentile(dm,97.5):.3f}]")

# ---- Target B: baseline-only vs early-change AUC ----
print(f"\n-- Target B: best AUC per set (exploratory, n={len(df_b)}) --")
for label, sets, res in [("B1 baseline-only", FEATURE_SETS_B1, results_b1),
                         ("B2 early-change",  FEATURE_SETS_B2, results_b2)]:
    print(f"  {label}:")
    for sn in sets:
        if sn in res:
            print(f"    {sn:<22} {best_name(res[sn], True):<18} "
                  f"AUC={best_auc(res[sn], True):.3f}")


## 8. Model interpretation and Target B figure


In [ ]:
# STEP 13 - SHAP interpretation of the selected specification (Figures 4 and 6)
# Computed on S2 (Clinical + FreeSurfer), the best primary specification by
# AUC and composite score, exactly as reported in the thesis.
import shap
from matplotlib.patches import Patch

model, X = fitted_a['S2_Clin_FS']
clf      = model.named_steps['clf']

# Transform X through the imputer + scaler (every pipeline step but the classifier)
X_t = pd.DataFrame(model[:-1].transform(X), columns=X.columns)

name_map = {
    'MMSE': 'MMSE', 'nWBV': 'nWBV', 'Age': 'Age', 'EDUC': 'Education',
    'sex': 'Sex', 'SES': 'SES', 'eTIV': 'eTIV', 'ASF': 'ASF',
}
X_display = X_t.rename(columns=name_map)

explainer   = shap.LinearExplainer(clf, X_t)
shap_values = explainer.shap_values(X_t)
shap_df     = pd.DataFrame(shap_values, columns=X_display.columns)

# ---- Figure 4: beeswarm (direction + magnitude of each feature) ----
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_df.values, X_display, show=False, alpha=0.5, plot_size=None)
plt.title('SHAP beeswarm - S2 (Clinical + FreeSurfer)\n'
          'Elastic Net LR | Target A: Demented vs Nondemented\n'
          'Red = high feature value   Blue = low feature value',
          fontsize=10, fontweight='bold', pad=12)
plt.xlabel('SHAP value  (positive -> pushes toward Demented, '
           'negative -> pushes toward Nondemented)', fontsize=9)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=200, bbox_inches='tight')
plt.show()

# ---- Figure 6: mean absolute SHAP, coloured by feature group ----
mean_abs = pd.Series(np.abs(shap_df.values).mean(axis=0),
                     index=X_display.columns).sort_values(ascending=True)
clinical   = {'MMSE', 'Age', 'Education', 'Sex', 'SES'}
freesurfer = {'nWBV', 'eTIV', 'ASF'}
colors = ['#4C72B0' if f in clinical else '#55A868' for f in mean_abs.index]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(mean_abs.index, mean_abs.values, color=colors,
               alpha=0.85, edgecolor='white', height=0.6)
for bar, val in zip(bars, mean_abs.values):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
ax.legend(handles=[Patch(facecolor='#4C72B0', alpha=0.85,
                         label='Clinical (Age, Sex, EDUC, SES, MMSE)'),
                   Patch(facecolor='#55A868', alpha=0.85,
                         label='FreeSurfer (nWBV, eTIV, ASF)')],
          fontsize=8, loc='lower right')
ax.set_xlabel('Mean |SHAP value|', fontsize=10)
ax.set_title('Mean absolute SHAP values by feature - S2\n'
             'Elastic Net LR | Target A: Demented vs Nondemented',
             fontsize=10, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=200, bbox_inches='tight')
plt.show()

# ---- Mean |SHAP| per feature, for the thesis text ----
print("\n  Mean |SHAP| per feature:")
for feat, val in mean_abs.sort_values(ascending=False).items():
    grp = 'Clinical' if feat in clinical else 'FreeSurfer'
    print(f"    {feat:<22} {val:>10.4f}  {grp}")


In [ ]:
# STEP 14 - Figure 5: Monte Carlo stability of the Target B early-change sets
# Reproduces the forest plot in the thesis from the mc_results computed above.
fig5_sets = {
    'B2_Clin_delta':     'B1  Clinical + d',
    'B2_Clin_FS_delta':  'B2  Clinical + FreeSurfer + d',
    'B2_Clin_MRI_delta': 'B3  Clinical + MRI proxy + d',
    'B2_All_delta':      'B4  Clinical + FreeSurfer + MRI proxy + d',
}
rows = [k for k in fig5_sets if k in mc_results and mc_results[k]['auc']]

fig, ax = plt.subplots(figsize=(8, 4))
for i, sn in enumerate(rows):
    aucs = np.array(mc_results[sn]['auc'])
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    ax.errorbar(aucs.mean(), i,
                xerr=[[aucs.mean() - lo], [hi - aucs.mean()]],
                fmt='o', color='#4C72B0', capsize=4, markersize=7)
ax.axvline(0.5, color='#4C72B0', ls='--', lw=1.2)   # chance level
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([fig5_sets[s] for s in rows], fontsize=9)
ax.set_xlabel('Mean AUC')
ax.set_title('Target B Monte Carlo stability analysis', fontsize=11)
ax.set_xlim(0.3, 1.0)
plt.tight_layout()
plt.savefig('fig_targetB_montecarlo.pdf', bbox_inches='tight')
plt.show()
